In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.model_selection import train_test_split

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bank-transaction-fraud-detection/Bank_Transaction_Fraud_Detection.csv


# **Understanding the Data**

In [2]:
transaction_df = pd.read_csv('/kaggle/input/bank-transaction-fraud-detection/Bank_Transaction_Fraud_Detection.csv')
pd.set_option('display.max_columns', 24)
transaction_df.head()

,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,Transaction_Time,Transaction_Amount,Merchant_ID,Transaction_Type,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
0,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Osha Tella,Male,60,Kerala,Thiruvananthapuram,Thiruvananthapuram Branch,Savings,4fa3208f-9e23-42dc-b330-844829d0c12c,23-01-2025,16:04:07,32415.45,214e03c5-5c34-40d1-a66c-f440aa2bbd02,Transfer,Restaurant,74557.27,Voice Assistant,"Thiruvananthapuram, Kerala",POS,0,INR,+9198579XXXXXX,Bitcoin transaction,oshaXXXXX@XXXXX.com
1,7c14ad51-781a-4db9-b7bd-67439c175262,Hredhaan Khosla,Female,51,Maharashtra,Nashik,Nashik Branch,Business,c9de0c06-2c4c-40a9-97ed-3c7b8f97c79c,11-01-2025,17:14:53,43622.60,f9e3f11f-28d3-4199-b0ca-f225a155ede6,Bill Payment,Restaurant,74622.66,POS Mobile Device,"Nashik, Maharashtra",Desktop,0,INR,+9191074XXXXXX,Grocery delivery,hredhaanXXXX@XXXXXX.com
2,3a73a0e5-d4da-45aa-85f3-528413900a35,Ekani Nazareth,Male,20,Bihar,Bhagalpur,Bhagalpur Branch,Savings,e41c55f9-c016-4ff3-872b-cae72467c75c,25-01-2025,03:09:52,63062.56,97977d83-5486-4510-af1c-8dada3e1cfa0,Bill Payment,Groceries,66817.99,ATM,"Bhagalpur, Bihar",Desktop,0,INR,+9197745XXXXXX,Mutual fund investment,ekaniXXX@XXXXXX.com
3,7902f4ef-9050-4a79-857d-9c2ea3181940,Yamini Ramachandran,Female,57,Tamil Nadu,Chennai,Chennai Branch,Business,7f7ee11b-ff2c-45a3-802a-49bc47c02ecb,19-01-2025,12:27:02,14000.72,f45cd6b3-5092-44d0-8afb-490894605184,Debit,Entertainment,58177.08,POS Mobile App,"Chennai, Tamil Nadu",Mobile,0,INR,+9195889XXXXXX,Food delivery,yaminiXXXXX@XXXXXXX.com
4,3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9,Kritika Rege,Female,43,Punjab,Amritsar,Amritsar Branch,Savings,f8e6ac6f-81a1-4985-bf12-f60967d852ef,30-01-2025,18:30:46,18335.16,70dd77dd-3b00-4b2c-8ebc-cfb8af5f6741,Transfer,Entertainment,16108.56,Virtual Card,"Amritsar, Punjab",Mobile,0,INR,+9195316XXXXXX,Debt repayment,kritikaXXXX@XXXXXX.com


In [3]:
transaction_df.columns

Index(['Customer_ID', 'Customer_Name', 'Gender', 'Age', 'State', 'City',
       'Bank_Branch', 'Account_Type', 'Transaction_ID', 'Transaction_Date',
       'Transaction_Time', 'Transaction_Amount', 'Merchant_ID',
       'Transaction_Type', 'Merchant_Category', 'Account_Balance',
       'Transaction_Device', 'Transaction_Location', 'Device_Type', 'Is_Fraud',
       'Transaction_Currency', 'Customer_Contact', 'Transaction_Description',
       'Customer_Email'],
      dtype='object')

In [4]:
"""There are 24 columns with 200,000 rows of non-null values. 
Series contain a mix of objects, floats, integers."""

transaction_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Customer_ID              200000 non-null  object 
 1   Customer_Name            200000 non-null  object 
 2   Gender                   200000 non-null  object 
 3   Age                      200000 non-null  int64  
 4   State                    200000 non-null  object 
 5   City                     200000 non-null  object 
 6   Bank_Branch              200000 non-null  object 
 7   Account_Type             200000 non-null  object 
 8   Transaction_ID           200000 non-null  object 
 9   Transaction_Date         200000 non-null  object 
 10  Transaction_Time         200000 non-null  object 
 11  Transaction_Amount       200000 non-null  float64
 12  Merchant_ID              200000 non-null  object 
 13  Transaction_Type         200000 non-null  object 
 14  Merc

In [5]:
transaction_df.shape

(200000, 24)

In [6]:
#Summary Stats 

"""
Age:
	•	The minimum age is 18, while the maximum age is 70 years old.
	•	On average, account holders are 44 years old.
	•	The standard deviation is 15, meaning that most account holders’ ages 
    will fall within ±15 years of the mean (typically between 29 and 59 years old). 
    If the dataset follows a normal distribution, about 68% of account holders are 
    between 29 and 59 years old (within one standard deviation from the mean).
	•	25% of account holders are aged 31 years old or younger.
	•	The median age (50%) is 44 years old, meaning half of the account holders 
    are 44 years old or younger.
	•	75% of account holders are aged 57 years old or younger.

Transaction Amount:
	•	The mean transaction amount is 49,538.02 INR, which closely aligns 
    with the median of 49,502.44 INR.
	•	The standard deviation of 28,551.87 INR indicates considerable variation 
    in transaction amounts.
	•	The distribution appears normal, as the mean and median are nearly identical.
	•	The minimum transaction amount is 11 INR, while the maximum transaction amount 
    is 99,000 INR, which is close to the maximum account balance, indicating that 
    high-value transactions occur.

Account Balance:
	•	The average account balance is 52,438 INR, reflecting the typical balance. 
    However, this figure can be skewed by extreme values, such as very high or very low balances.
	•	The standard deviation of 28,552 INR suggests significant spread in the 
    account balances. In a normal distribution, about 68% of account balances 
    will fall within ±28,552 INR of the mean, i.e., between 23,086 INR and 81,990 INR.
	•	25% of account balances are ≤ 28,552.87 INR.
	•	The 50th percentile (median) of account balances is 52,372.56 INR.
	•	75% of account balances are ≤ 76,147.67 INR.
	•	The minimum account balance is 5,001 INR, while the maximum balance is 
    99,999.95 INR, which could suggest a maximum deposit limit for the account.

Is_Fraud:
	•	The mean for the Is_Fraud column is 0.050440, which means approximately 
    5.04% of transactions are flagged as fraudulent.
	•	Based on this, you would expect about 1,008 fraudulent transactions in a 
    dataset of 20,000 rows.
	•	However, the value_counts() output suggests the dataset contains 200,000 
    rows (189,912 non-fraudulent and 10,088 fraudulent), which indicates an imbalance 
    in the dataset. Further exploratory data analysis (EDA) is needed to resolve this discrepancy.

Further Notes:
	•	The account balances’ distribution and high standard deviation suggest 
    there might be accounts with extreme balances.
	•	Fraud detection needs to consider the imbalance in the dataset, which may 
    affect model training.
"""

transaction_df.describe()

,Age,Transaction_Amount,Account_Balance,Is_Fraud
count,200000.000000,200000.000000,200000.000000,200000.000000
mean,44.015110,49538.015554,52437.988784,0.050440
std,15.288774,28551.874004,27399.507128,0.218852
min,18.000000,10.290000,5000.820000,0.000000
25%,31.000000,24851.345000,28742.395000,0.000000
50%,44.000000,49502.440000,52372.555000,0.000000
75%,57.000000,74314.625000,76147.670000,0.000000
max,70.000000,98999.980000,99999.950000,1.000000


In [7]:
"""There are 34 unique states in the database and 145 different cities/branches. 
The dataset includes three unique account types. Transactions have occurred on 
only 31 distinct dates, which seems unusual.

There are 5 different transaction types and 6 merchant categories, along with 
20 unique transaction devices and 148 transaction locations. Additionally, the dataset 
contains 4 different device types.

We have 4,779 unique customer emails but 142,699 customer names, which appears inconsistent. 
There are also 172 different types of transaction descriptions. Further exploratory 
data analysis (EDA) is needed to investigate these insights and anomalies."""

transaction_df.describe(include='all')

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,Transaction_Time,Transaction_Amount,Merchant_ID,Transaction_Type,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
count,200000,200000,200000,200000.000000,200000,200000,200000,200000,200000,200000,200000,200000.000000,200000,200000,200000,200000.000000,200000,200000,200000,200000.000000,200000,200000,200000,200000
unique,200000,142699,2,NaN,34,145,145,3,200000,31,77856,NaN,200000,5,6,NaN,20,148,4,NaN,1,9000,172,4779
top,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Aahana Kala,Male,NaN,Nagaland,Chandigarh,Chandigarh Branch,Checking,4fa3208f-9e23-42dc-b330-844829d0c12c,29-01-2025,07:30:31,NaN,214e03c5-5c34-40d1-a66c-f440aa2bbd02,Credit,Restaurant,NaN,Self-service Banking Machine,"Kavaratti, Lakshadweep",POS,NaN,INR,+9191471XXXXXX,Sports ticket,krishnaXXX@XXXXX.com
freq,1,8,100452,NaN,6031,8135,8135,66924,1,6854,11,NaN,1,40180,33525,NaN,21707,5954,50111,NaN,200000,41,1268,99
mean,NaN,NaN,NaN,44.015110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49538.015554,NaN,NaN,NaN,52437.988784,NaN,NaN,NaN,0.050440,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,15.288774,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28551.874004,NaN,NaN,NaN,27399.507128,NaN,NaN,NaN,0.218852,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.290000,NaN,NaN,NaN,5000.820000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24851.345000,NaN,NaN,NaN,28742.395000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,44.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49502.440000,NaN,NaN,NaN,52372.555000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,57.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74314.625000,NaN,NaN,NaN,76147.670000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN


In [8]:
transaction_df["State"].value_counts(sort = True)

State
Nagaland                                    6031
Meghalaya                                   6003
Uttar Pradesh                               6002
Uttarakhand                                 5985
Lakshadweep                                 5954
Telangana                                   5952
Haryana                                     5947
Delhi                                       5943
Kerala                                      5933
Madhya Pradesh                              5928
Arunachal Pradesh                           5919
Punjab                                      5912
Gujarat                                     5901
Odisha                                      5899
Jharkhand                                   5898
Mizoram                                     5892
Himachal Pradesh                            5875
Goa                                         5871
Tripura                                     5869
Manipur                                     5869
Bihar         

In [9]:
pd.set_option('display.max_rows', 145)
transaction_df["City"].value_counts(sort = False)

City
Thiruvananthapuram    1140
Nashik                1125
Bhagalpur             1139
Chennai               1161
Amritsar              1172
Ahmedabad             1229
New Delhi             1150
Port Blair            1950
Bhopal                1174
Jagdalpur             1132
Vadodara              1153
Chandigarh            8135
Champhai              1469
Korba                 1159
Kolkata               1165
Gaya                  1138
Jorethang             1481
Silvassa              1916
Kanpur                1178
Nagpur                1180
Bhubaneswar           1186
Ambassa               1382
Jorhat                1209
Diglipur              1926
Salem                 1135
Durg                  1142
Churachandpur         1502
Kottayam              1243
Varanasi              1239
Imphal                1467
Belgaum               1156
Agra                  1203
Durgapur              1181
Daman                 2022
Bilaspur              1240
Nellore               1150
Patiala               1

In [10]:
"""Women are slightly underrepresented as account holders in our dataset, 
with a difference of 0.5 percentage points between male and female account holders."""


print(transaction_df["Gender"].value_counts())
gender_ratio = transaction_df["Gender"].value_counts() / len(transaction_df["Gender"].index)
gender_ratio


Gender
Male      100452
Female     99548
Name: count, dtype: int64


Gender
Male      0.50226
Female    0.49774
Name: count, dtype: float64

In [11]:
"""There are three types of accounts: Checking, Savings, and Business."""

transaction_df["Account_Type"].value_counts()

Account_Type
Checking    66924
Savings     66593
Business    66483
Name: count, dtype: int64

In [12]:
""""""

transaction_df.groupby('Account_Type').describe().unstack(1)

                           Account_Type
Age                 count  Business        66483.000000
                           Checking        66924.000000
                           Savings         66593.000000
                    mean   Business           44.016606
                           Checking           44.022623
                           Savings            44.006067
                    std    Business           15.268000
                           Checking           15.272780
                           Savings            15.325747
                    min    Business           18.000000
                           Checking           18.000000
                           Savings            18.000000
                    25%    Business           31.000000
                           Checking           31.000000
                           Savings            31.000000
                    50%    Business           44.000000
                           Checking           44.000000
        

In [13]:
"""Dataset has no missing values"""
transaction_df.isna().sum()

Customer_ID                0
Customer_Name              0
Gender                     0
Age                        0
State                      0
City                       0
Bank_Branch                0
Account_Type               0
Transaction_ID             0
Transaction_Date           0
Transaction_Time           0
Transaction_Amount         0
Merchant_ID                0
Transaction_Type           0
Merchant_Category          0
Account_Balance            0
Transaction_Device         0
Transaction_Location       0
Device_Type                0
Is_Fraud                   0
Transaction_Currency       0
Customer_Contact           0
Transaction_Description    0
Customer_Email             0
dtype: int64

In [14]:
"""There are about 10,000 cases of fraud in our 200,000 row-dataset."""

fraud_count = transaction_df.value_counts(subset= 'Is_Fraud')
fraud_count


Is_Fraud
0    189912
1     10088
Name: count, dtype: int64

In [15]:
"""Based on this sample, we observe a 5% fraud rate, which is notably higher than the 
global average of approximately 0.1% in banking. This provides us with more fraud examples to train 
our model, improving its ability to detect fraud patterns. However, the class imbalance—where 
95% of the transactions are non-fraudulent—raises the risk of false negatives. 
This imbalance could lead to the model being biased toward predicting non-fraudulent 
transactions, potentially missing some fraudulent ones. Handling this imbalance through 
techniques such as resampling or class weighting will be crucial to improving model accuracy 
and reducing false negatives."""

ratio_cases = fraud_count/len(transaction_df.index)
ratio_cases

Is_Fraud
0    0.94956
1    0.05044
Name: count, dtype: float64